In [15]:
import pandas as pd
import os,re

os.chdir('J:\\CMIP6_r1i1p1f1\\zonal_mean_df')

In [16]:
def add_daily_dates(group):
    group['time'] = pd.to_datetime(group['time'], errors='coerce')
    
    year_month = group['time'].iloc[0].strftime('%Y-%m')
    start_date = pd.to_datetime(year_month + "-01")
    end_date = pd.to_datetime((start_date + pd.offsets.MonthEnd(0)))

    date_range = pd.date_range(start=start_date, end=end_date, freq='D')[:len(group)]
    group['time'] = date_range
    return group


In [17]:
csv_list = [f for f in os.listdir() if f.endswith('.csv')  and 'r1i1p1f1' in f]
prefix_list = ['_'.join(item.split('_')[:5]) for item in csv_list]
prefix_list = [item for item in prefix_list if 'r2i1p1f1' not in item]
print(prefix_list)

['pr_Amon_ACCESS-CM2_ssp245_r1i1p1f1', 'pr_Amon_ACCESS-ESM1-5_historical_r1i1p1f1', 'pr_Amon_ACCESS-ESM1-5_ssp245_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1', 'pr_Amon_AWI-CM-1-1-

In [18]:
import pandas as pd

# Area mapping
area_dict = {
    "dsk": 18658.8,
    "hsg": 4316.58,
    "kq": 46986.12,
    "slglk": 19275.39,
    "tgzlk": 14811.93,
    "wlwt": 38722.68,
    "xhl": 12948.48
}

# Choose power (example: p = 1 for linear weighting)
p = 1
weights = {k: v**p for k, v in area_dict.items()}

# Convert to Series for alignment
weight_series = pd.Series(weights)

In [19]:
for prefix in prefix_list:
    [var,frq,gcm,ssp,configuration ] = prefix.split('_')
    export_file_name = "_".join([var, gcm, ssp, configuration]) + ".csv"
    if configuration != 'r1i1p1f1':
        continue
        print(prefix.split('_')[4])

    # Check if the file exists
    if not os.path.exists(os.path.join('mon', export_file_name)):
        print(prefix, 'not exist')
            # Define the prefix to match
        # prefix = 'tas_day_TaiESM1_ssp585_r1i1p1f1_gn_20350101-20441231'

        # Get a list of files starting with the specified prefix
        file_list = [file for file in os.listdir() if file.startswith(prefix) and file.endswith('.csv') and file.split('_')[4] == 'r1i1p1f1']

        # Read and concatenate all the matching CSV files
        combined_df = pd.concat([pd.read_csv(file) for file in file_list], ignore_index=True)

        # combined_df = combined_df.groupby("time", group_keys=False).apply(add_daily_dates)

        combined_df = combined_df.sort_values(by="time", ascending=True)
        print(combined_df.head())
        if frq == 'Amon':
            combined_df['time'] = pd.to_datetime(combined_df['time'], format="mixed")
            combined_df.to_csv(os.path.join('mon', export_file_name), index=False)
        else:
            combined_df['time'] = pd.to_datetime(combined_df['time'], format="mixed")
            combined_df = combined_df.set_index('time')
            combined_df=combined_df.resample('M').mean()
        # combined_df['time'] = pd.to_datetime(combined_df['time'], format="mixed")
        # combined_df.set_index('time',inplace=True)
        combined_df.to_csv(os.path.join('mon', export_file_name))

    else:
        print(f"{prefix} already exists")


pr_Amon_ACCESS-CM2_ssp245_r1i1p1f1 already exists
pr_Amon_ACCESS-ESM1-5_historical_r1i1p1f1 not exist
         time       dsk       hsg        kq     slglk     tgzlk      wlwt  \
0  1850-01-16  0.000002  0.000002  0.000003  0.000001  0.000008  0.000006   
1  1850-02-15  0.000003  0.000002  0.000014  0.000017  0.000019  0.000019   
2  1850-03-16  0.000012  0.000006  0.000039  0.000023  0.000029  0.000035   
3  1850-04-16  0.000030  0.000017  0.000059  0.000071  0.000065  0.000072   
4  1850-05-16  0.000029  0.000044  0.000069  0.000049  0.000070  0.000085   

            xhl  
0  8.642068e-07  
1  1.009660e-05  
2  3.017976e-05  
3  3.987549e-05  
4  3.032722e-05  
pr_Amon_ACCESS-ESM1-5_ssp245_r1i1p1f1 already exists
pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1 not exist
         time       dsk       hsg        kq     slglk     tgzlk      wlwt  \
0  1850-01-16  0.000006  0.000006  0.000012  0.000004  0.000006  0.000004   
1  1850-02-15  0.000011  0.000010  0.000011  0.000008  0.000011  0.0

In [20]:
import glob ,re

# Search in current directory
mon_station_list = glob.glob("mon/pr_*_ssp245_r1i1p1f1.csv") + glob.glob("mon/pr_*_historical_r1i1p1f1.csv")

for csv_file in mon_station_list:
    mon_station_df = pd.read_csv(csv_file, index_col=0)
    print(csv_file)
    [frq ,var,gcm,ssp,configuration ] = re.split(r'[\\_]', csv_file)


mon\pr_ACCESS-CM2_ssp245_r1i1p1f1.csv
mon\pr_ACCESS-ESM1-5_ssp245_r1i1p1f1.csv
mon\pr_AWI-CM-1-1-MR_ssp245_r1i1p1f1.csv
mon\pr_BCC-CSM2-MR_ssp245_r1i1p1f1.csv
mon\pr_CanESM5-1_ssp245_r1i1p1f1.csv
mon\pr_EC-Earth3-Veg_ssp245_r1i1p1f1.csv
mon\pr_EC-Earth3_ssp245_r1i1p1f1.csv
mon\pr_FGOALS-g3_ssp245_r1i1p1f1.csv
mon\pr_IITM-ESM_ssp245_r1i1p1f1.csv
mon\pr_INM-CM4-8_ssp245_r1i1p1f1.csv
mon\pr_INM-CM5-0_ssp245_r1i1p1f1.csv
mon\pr_IPSL-CM6A-LR_ssp245_r1i1p1f1.csv
mon\pr_MIROC6_ssp245_r1i1p1f1.csv
mon\pr_MPI-ESM1-2-HR_ssp245_r1i1p1f1.csv
mon\pr_MRI-ESM2-0_ssp245_r1i1p1f1.csv
mon\pr_NorESM2-LM_ssp245_r1i1p1f1.csv
mon\pr_TaiESM1_ssp245_r1i1p1f1.csv
mon\pr_ACCESS-ESM1-5_historical_r1i1p1f1.csv
mon\pr_AWI-CM-1-1-MR_historical_r1i1p1f1.csv
mon\pr_BCC-CSM2-MR_historical_r1i1p1f1.csv
mon\pr_CanESM5-1_historical_r1i1p1f1.csv
mon\pr_CanESM5_historical_r1i1p1f1.csv
mon\pr_EC-Earth3-Veg_historical_r1i1p1f1.csv
mon\pr_EC-Earth3_historical_r1i1p1f1.csv
mon\pr_FGOALS-g3_historical_r1i1p1f1.csv
mon\pr_GISS-E

In [21]:
mon_station_list

['mon\\pr_ACCESS-CM2_ssp245_r1i1p1f1.csv',
 'mon\\pr_ACCESS-ESM1-5_ssp245_r1i1p1f1.csv',
 'mon\\pr_AWI-CM-1-1-MR_ssp245_r1i1p1f1.csv',
 'mon\\pr_BCC-CSM2-MR_ssp245_r1i1p1f1.csv',
 'mon\\pr_CanESM5-1_ssp245_r1i1p1f1.csv',
 'mon\\pr_EC-Earth3-Veg_ssp245_r1i1p1f1.csv',
 'mon\\pr_EC-Earth3_ssp245_r1i1p1f1.csv',
 'mon\\pr_FGOALS-g3_ssp245_r1i1p1f1.csv',
 'mon\\pr_IITM-ESM_ssp245_r1i1p1f1.csv',
 'mon\\pr_INM-CM4-8_ssp245_r1i1p1f1.csv',
 'mon\\pr_INM-CM5-0_ssp245_r1i1p1f1.csv',
 'mon\\pr_IPSL-CM6A-LR_ssp245_r1i1p1f1.csv',
 'mon\\pr_MIROC6_ssp245_r1i1p1f1.csv',
 'mon\\pr_MPI-ESM1-2-HR_ssp245_r1i1p1f1.csv',
 'mon\\pr_MRI-ESM2-0_ssp245_r1i1p1f1.csv',
 'mon\\pr_NorESM2-LM_ssp245_r1i1p1f1.csv',
 'mon\\pr_TaiESM1_ssp245_r1i1p1f1.csv',
 'mon\\pr_ACCESS-ESM1-5_historical_r1i1p1f1.csv',
 'mon\\pr_AWI-CM-1-1-MR_historical_r1i1p1f1.csv',
 'mon\\pr_BCC-CSM2-MR_historical_r1i1p1f1.csv',
 'mon\\pr_CanESM5-1_historical_r1i1p1f1.csv',
 'mon\\pr_CanESM5_historical_r1i1p1f1.csv',
 'mon\\pr_EC-Earth3-Veg_histor

In [22]:
mon_station_df

,time,dsk,hsg,kq,slglk,tgzlk,wlwt,xhl
0,1950-01-16,0.000004,4.526572e-06,0.000004,7.145299e-06,1.194668e-06,2.147350e-06,0.000005
1,1950-02-15,0.000003,3.094278e-06,0.000009,5.433275e-06,4.618210e-06,5.442247e-06,0.000004
2,1950-03-16,0.000005,4.154404e-06,0.000014,1.278252e-05,2.967477e-06,3.602161e-06,0.000007
3,1950-04-16,0.000005,4.175285e-06,0.000026,1.870100e-05,1.467944e-05,1.391236e-05,0.000013
4,1950-05-16,0.000009,6.497820e-06,0.000047,4.921179e-05,3.869665e-05,2.614646e-05,0.000015
...,...,...,...,...,...,...,...,...
355,1989-08-16,0.000001,1.994778e-07,0.000001,8.857049e-07,7.974944e-07,2.524560e-06,0.000002
356,1989-09-16,0.000003,3.837102e-06,0.000001,1.011603e-07,5.523506e-07,9.309569e-07,0.000003
357,1989-10-16,0.000002,2.372781e-06,0.000008,3.261040e-06,2.209876e-06,9.328226e-07,0.000003
358,1989-11-16,0.000002,1.779220e-06,0.000005,5.055415e-06,2.003739e-06,3.495247e-06,0.000007


In [72]:
import glob, re
import pandas as pd

# Area mapping
area_dict = {
    "dsk": 18658.8,
    "hsg": 4316.58,
    "kq": 46986.12,
    "slglk": 19275.39,
    "tgzlk": 14811.93,
    "wlwt": 38722.68,
    "xhl": 12948.48
}

# Choose power (example: p = 1 for linear weighting)
p = 1
weights = {k: v**p for k, v in area_dict.items()}
weight_series = pd.Series(weights)

# Search for files
mon_station_list =  glob.glob("mon/pr_*_historical_r1i1p1f1.csv")+glob.glob("mon/pr_*_ssp245_r1i1p1f1.csv")

# Container for results
results = {}

gcm_temporary_list = []
for csv_file in mon_station_list:
    # Read file
    mon_station_df = pd.read_csv(csv_file, index_col=0)
    mon_station_df.index = pd.to_datetime(mon_station_df.index)   # ensure datetime


    # Extract metadata
    frq, var, gcm, ssp, configuration = re.split(r'[\\_]', csv_file)

    if "time" in mon_station_df.columns:
        # Convert to datetime if not already
        mon_station_df["time"] = pd.to_datetime(mon_station_df["time"], errors="coerce")

        # Set as index
        mon_station_df = mon_station_df.set_index("time").sort_index()
    mon_station_df.index = mon_station_df.index.to_period("M")    # convert to year-month

    # Compute weighted average time series
    weighted_avg = (mon_station_df * weight_series).sum(axis=1) / weight_series.sum()

    if gcm in results:
        # Concatenate existing series with the new one
        results[gcm] = pd.concat([results[gcm], weighted_avg])
    else:
        # First time, just assign
        results[gcm] = weighted_avg

    # Ensure it's sorted by time index
    results[gcm] = results[gcm].sort_index()

    # if gcm in gcm_temporary_list:
    #     print(gcm)
    # else:
    #     gcm_temporary_list.append(gcm)
    #     # Store in dictionary keyed by GCM
    #     results[gcm] = weighted_avg
# Combine into one DataFrame
weighted_mon_df = pd.DataFrame(results)

weighted_mon_df

,ACCESS-ESM1-5,AWI-CM-1-1-MR,BCC-CSM2-MR,CanESM5-1,CanESM5,EC-Earth3-Veg,EC-Earth3,FGOALS-g3,GISS-E2-1-G,INM-CM4-8,INM-CM5-0,IPSL-CM6A-LR,MIROC6,MPI-ESM1-2-HR,MRI-ESM2-0,NorESM2-LM,ACCESS-CM2,IITM-ESM,TaiESM1
time,,,,,,,,,,,,,,,,,,,
1850-01,0.000004,0.000007,0.000006,0.000004,0.000007,0.000005,0.000003,0.000007,0.000021,0.000008,0.000008,NaN,NaN,0.000009,0.000006,NaN,NaN,NaN,NaN
1850-02,0.000014,0.000010,0.000013,0.000006,0.000007,0.000008,0.000004,0.000012,0.000030,0.000010,0.000012,NaN,NaN,0.000013,0.000006,NaN,NaN,NaN,NaN
1850-03,0.000030,0.000012,0.000021,0.000011,0.000021,0.000011,0.000007,0.000032,0.000028,0.000015,0.000016,NaN,NaN,0.000021,0.000024,NaN,NaN,NaN,NaN
1850-04,0.000058,0.000030,0.000029,0.000012,0.000020,0.000008,0.000012,0.000037,0.000031,0.000033,0.000023,NaN,NaN,0.000016,0.000030,NaN,NaN,NaN,NaN
1850-05,0.000062,0.000028,0.000015,0.000012,0.000021,0.000016,0.000015,0.000046,0.000019,0.000039,0.000046,NaN,NaN,0.000029,0.000048,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100-08,0.000033,0.000012,0.000027,0.000011,NaN,0.000022,0.000019,0.000042,NaN,0.000011,0.000010,NaN,0.000037,0.000007,0.000030,5.802673e-06,0.000031,NaN,0.000011
2100-09,0.000028,0.000008,0.000011,0.000006,NaN,0.000014,0.000015,0.000036,NaN,0.000008,0.000011,NaN,0.000028,0.000026,0.000016,1.610374e-06,0.000038,NaN,0.000002
2100-10,0.000020,0.000003,0.000016,0.000005,NaN,0.000009,0.000009,0.000032,NaN,0.000011,0.000011,NaN,0.000021,0.000015,0.000009,5.747078e-07,0.000009,NaN,0.000007


In [73]:
weighted_mon_df.to_csv('prcp_2015_2100_mon.csv')